# Testing with my past example

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Dict

class ReviewClassification(BaseModel):
    classification: str = Field(..., description = "Topic of the review")
    classification_explanation: str = Field(..., description = "explanation of why the classification rating was given")
    sentiment: int = Field(..., description = "1-5 rating (1 very negative to 5 very positive) for the topic")
    sentiment_explanation: str =  Field(..., description = "explanation of why the sentiment rating was given")

topics = []
template = f"""
You are a expert customer testimonial reviewer who loves to read a customer's review for a given product to identify topics.
Your task is to read a given review: classify the topic and rate the sentiment
"""
customer_review = "As the title says, this is simply the best backpack I've ever used. It's extremely comfortable and sometimes I feel like I forgot to pack my laptop and lunch it's that light. \r\n\r\nMy friends think it's a waste of money (they're not totally wrong) but even when they tried it on they agreed that its simply better than anything they've used. \r\n\r\nI'm a small male who's about 169cm ±2cm and very skinny at 45kg, it's a tad big for me but still fits quite nicely when pulling the adjustable shoulder straps nearly all the way in. \r\n\r\nIt fits exactly a framework laptop, lunch box, pencil case, glasses case, work uniform, whole old backpack, 40oz LTT water bottle, keys and other small things in the front pouch and side pocket. \r\n\r\nThe verdict: if you're just as, if not smaller than me I'd highly recommend waiting for the Yvonne sized one as Linus mentioned a wan or 3 ago (and showed off in the FP exclusive BTS of the AMD upgrade thing Yvonne was in recently) but if you value products that are going to last a long while and you like LTT there simply isn't a better bag for the (albeit quite high) price. (PS Linus I know you read these sometimes and in which case hello!)"
input_text = f""" Given this customer review:
{customer_review}
"""

In [ ]:
import json
from openai import OpenAI

client = OpenAI(
    base_url = 'http://localhost:11434/v1',
    api_key='ollama', # required, but unused
)

json_completion = client.beta.chat.completions.parse(
    model="gemma2:9b-instruct-q8_0",

    response_format= ReviewClassification,
    messages=[
            {"role": "user", "content": template},           
            {"role": "user", "content": input_text},
    ],
    temperature=0.02,
    )
ReviewClassification.model_validate(json_completion.choices[0].message.parsed)

completion = json_completion.to_dict()
print(json.dumps(completion, indent=1))
json_completion.choices[0].message.parsed.model_dump()

{
 "id": "chatcmpl-773",
 "choices": [
  {
   "finish_reason": "stop",
   "index": 0,
   "message": {
    "content": "{\"classification\": \"Product Review\", \"classification_explanation\": \"The review focuses on the user's experience with a specific backpack, detailing its features, pros, cons, and overall recommendation.\", \"sentiment\": 4, \"sentiment_explanation\": \"The sentiment is overwhelmingly positive. The reviewer uses phrases like 'best backpack I've ever used', 'extremely comfortable', 'simply better than anything they've used', and 'highly recommend'. While acknowledging the price might be high and suggesting a smaller size for certain users, the overall tone is enthusiastic and appreciative.\"} ",
    "role": "assistant",
    "tool_calls": null,
    "parsed": {
     "classification": "Product Review",
     "classification_explanation": "The review focuses on the user's experience with a specific backpack, detailing its features, pros, cons, and overall recommendation.

{'classification': 'Product Review',
 'classification_explanation': "The review focuses on the user's experience with a specific backpack, detailing its features, pros, cons, and overall recommendation.",
 'sentiment': 4,
 'sentiment_explanation': "The sentiment is overwhelmingly positive. The reviewer uses phrases like 'best backpack I've ever used', 'extremely comfortable', 'simply better than anything they've used', and 'highly recommend'. While acknowledging the price might be high and suggesting a smaller size for certain users, the overall tone is enthusiastic and appreciative."}

In [ ]:
# --- deps ---
import dspy
from dspy.adapters.baml_adapter import BAMLAdapter
from pydantic import BaseModel

# ---------- Pydantic schema (for post-validate / typing) ----------
class ReviewClassification(BaseModel):
    classification: str                          # topic
    classification_explanation: str              # why that topic
    sentiment: int                               # 1..5
    sentiment_explanation: str                   # why that sentiment

# ---------- DSPy signature (structured I/O contract) ----------
class ReviewQA(dspy.Signature):
    """You are a expert customer testimonial reviewer who loves to read a customer's review for a given product to identify topics. Your task is to read a given review: classify the topic and rate the sentiment"""
    review: str = dspy.InputField(desc="Customer review text")

    classification: str = dspy.OutputField(desc="Topic of the review")
    classification_explanation: str = dspy.OutputField(desc="explanation of why the classification rating was given")
    sentiment: int = dspy.OutputField(desc="1-5 rating (1 very negative to 5 very positive) for the topic")
    sentiment_explanation: str = dspy.OutputField(desc="explanation of why the sentiment rating was given")

# ---------- LM config: use OpenAI adapter pointed at Ollama /v1 ----------
lm = dspy.LM(
    model="openai/gemma2:9b-instruct-q8_0",             # use 'openai/*' so usage is populated
    api_base="http://localhost:11434/v1",    # Ollama OpenAI-compatible endpoint
    api_key="ollama",                        # dummy; Ollama ignores it
    temperature=0.02,
    max_tokens=800,
    # If you later enable streaming elsewhere, this helps keep usage attached:
    stream_options={"include_usage": True},
)

# BAMLAdapter improves structured output reliability
dspy.configure(lm=lm, adapter=BAMLAdapter(), track_usage=True, cache=False)

# ---------- Run ----------
predict = dspy.Predict(ReviewQA)

customer_review = "As the title says, this is simply the best backpack I've ever used. It's extremely comfortable and sometimes I feel like I forgot to pack my laptop and lunch it's that light. \r\n\r\nMy friends think it's a waste of money (they're not totally wrong) but even when they tried it on they agreed that its simply better than anything they've used. \r\n\r\nI'm a small male who's about 169cm ±2cm and very skinny at 45kg, it's a tad big for me but still fits quite nicely when pulling the adjustable shoulder straps nearly all the way in. \r\n\r\nIt fits exactly a framework laptop, lunch box, pencil case, glasses case, work uniform, whole old backpack, 40oz LTT water bottle, keys and other small things in the front pouch and side pocket. \r\n\r\nThe verdict: if you're just as, if not smaller than me I'd highly recommend waiting for the Yvonne sized one as Linus mentioned a wan or 3 ago (and showed off in the FP exclusive BTS of the AMD upgrade thing Yvonne was in recently) but if you value products that are going to last a long while and you like LTT there simply isn't a better bag for the (albeit quite high) price. (PS Linus I know you read these sometimes and in which case hello!)"

res = predict(review=customer_review)

# Structured fields
print("classification:", res.classification)
print("sentiment:", res.sentiment)
print("classification_explanation:", res.classification_explanation[:200], "...")
print("sentiment_explanation:", res.sentiment_explanation[:200], "...")

# Token usage (prompt / completion / total)
print("usage:")
res.get_lm_usage()


classification: Product Review
sentiment: 5
classification_explanation: The text focuses on providing a detailed opinion and experience with a specific backpack. ...
sentiment_explanation: The review expresses overwhelmingly positive sentiment. The user repeatedly uses words like 'best,' 'comfortable,' 'simply better,' and 'highly recommend.' ...
usage:


{'openai/gemma2:9b-instruct-q8_0': {'completion_tokens': 76,
  'prompt_tokens': 581,
  'total_tokens': 657,
  'completion_tokens_details': None,
  'prompt_tokens_details': None}}

In [ ]:
dspy.inspect_history()





[2025-09-08T22:30:25.105659]

System message:

Your input fields are:
1. `review` (str):
Your output fields are:
1. `classification` (str):
2. `classification_explanation` (str):
3. `sentiment` (int):
4. `sentiment_explanation` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## review ## ]]
{review}

[[ ## classification ## ]]
Output field `classification` should be of type: string

[[ ## classification_explanation ## ]]
Output field `classification_explanation` should be of type: string

[[ ## sentiment ## ]]
Output field `sentiment` should be of type: int

[[ ## sentiment_explanation ## ]]
Output field `sentiment_explanation` should be of type: string

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are a expert customer testimonial reviewer who loves to read a customer's review for a given product to identify topics. Your task is to read a given review: classify the topic and rate th

In [ ]:
657 - 471

186